# Task 6: TNM Staging — Getotron Base
<!-- - **Model A**: `yikuan8/Clinical-Longformer` (max_len 1024/512) -->
- **Model B**: `UFNLP/gatortron-base` (512 tokens (345 million parameters))
- Both trained separately; probabilities averaged at inference
- Single T4 GPU (15 GB) with gradient-checkpointing + sequential training



## 1. Install

In [1]:
# %%capture
# # Pin pandas and scikit-learn alongside numpy to prevent binary incompatibility (Numpy 1.x vs 2.x)
# !pip install -q numpy==1.26.4 pandas==2.1.4 scikit-learn==1.3.2 scipy==1.13.0
# !pip install -q transformers==4.40.2 accelerate sentencepiece protobuf datasets torch seaborn matplotlib huggingface_hub anthropic
!pip install transformers==4.40.2 accelerate sentencepiece --force-reinstall -q
!pip install torch seaborn matplotlib huggingface_hub anthropic
!pip install protobuf
!pip install -q "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.5.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.4.4 which is incompatible.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2026.3.0 which is incompatible.
datasets 4.8.3 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2

## 2. Imports & GPU Setup

In [2]:
import os, re, json, warnings, gc, time, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)
    import random; random.seed(SEED + worker_id)

def make_loader_generator():
    g = torch.Generator()
    g.manual_seed(SEED)
    return g


# Force single GPU even if Kaggle shows 2
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.cuda.set_device(0)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def vram_stats():
    if not torch.cuda.is_available(): return 'no GPU'
    alloc  = torch.cuda.memory_allocated()/1e9
    reserv = torch.cuda.memory_reserved()/1e9
    total  = torch.cuda.get_device_properties(0).total_memory/1e9
    free   = total - reserv
    return f'alloc={alloc:.2f}GB  reserved={reserv:.2f}GB  free={free:.2f}GB/{total:.1f}GB'

def free_vram(label=''):
    """Aggressively free GPU memory between training runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
    if label:
        print(f'  [{label}] VRAM after free -> {vram_stats()}')

print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {vram_stats()}')


Device : cuda
GPU    : Tesla T4
VRAM   : alloc=0.00GB  reserved=0.00GB  free=15.64GB/15.6GB


## 3. Load Data

In [3]:
all_csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
print('CSVs found:'); [print(' ', c) for c in all_csvs]

def find_csv(keyword):
    m = [c for c in all_csvs if keyword.lower() in os.path.basename(c).lower()]
    if not m:
        raise FileNotFoundError(f'No CSV matching "{keyword}". Add your dataset via Add Data.')
    return m[0]

TRAIN_CSV = find_csv('train')
SUB_CSV   = find_csv('no_res')
TEST_CSV  = find_csv('test_hard')
print(f'\nTRAIN : {TRAIN_CSV}')
print(f'SUB   : {SUB_CSV}')
print(f'TEST   : {TEST_CSV}')

df_all = pd.read_csv(TRAIN_CSV)
df_sub = pd.read_csv(SUB_CSV)
# df_test = pd.read_csv(TEST_CSV)
df_test = pd.read_csv(TEST_CSV)

print(f'\nTrain shape : {df_all.shape}')
print(f'Sub shape   : {df_sub.shape}')
for col in ['t','n','m']:
    dist  = df_all[col].value_counts().sort_index().to_dict()
    nulls = df_all[col].isna().sum()
    print(f'  {col}: {dist}  nulls={nulls}')


CSVs found:
  /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/tcga_tnm_train.csv
  /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/tcga_tnm_val_no_res.csv
  /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/Task6_test.csv
  /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/Task6_test_hard.csv

TRAIN : /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/tcga_tnm_train.csv
SUB   : /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/tcga_tnm_val_no_res.csv
TEST   : /kaggle/input/datasets/spartacus00/tcga-reports/TCGA_Reports/Task6_test_hard.csv

Train shape : (6774, 5)
Sub shape   : (2279, 2)
  t: {1.0: 1484, 2.0: 1985, 3.0: 1795, 4.0: 589}  nulls=921
  n: {0.0: 2829, 1.0: 1241, 2.0: 589, 3.0: 167}  nulls=1948
  m: {0.0: 3650, 1.0: 266}  nulls=2858


## 4. Config — Two Models for Ensemble

In [4]:
# ── Shared hyper-parameters ───────────────────────────────────────────────
CONFIG = {
    # max_len: 1024 tokens handles full clinical reports (BigBird & Longformer
    # both support up to 4096, but 1024 is the sweet spot for T4 VRAM)
    'max_len'     : {'t': 1024, 'n': 1024, 'm': 512},
    'val_frac'    : 0.15,
    'epochs'      : 10,       # fixed — all tasks use 10 epochs
    'batch_size'  : 4,
    'grad_accum'  : 8,        # effective batch = 32
    'lr'          : 2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.06,
    'dropout'     : 0.10,
    'focal_gamma' : 2.0,
    'label_smooth': 0.05,
    'patience'    : 3,        # increased from 2 — less aggressive early stop
    'fp16'        : True,
    'grad_ckpt'   : True,
    'num_workers' : 2,
}

# ── Two long-document clinical models ─────────────────────────────────────
# MODEL_A = 'yikuan8/Clinical-BigBird'     # random block sparse attention
MODEL_A = 'yikuan8/Clinical-Longformer'
MODEL_B = 'UFNLP/gatortron-base'   # was 'yikuan8/Clinical-Longformer'  # sliding-window + global attention

# Ensemble weight (0.5/0.5 = simple average; tune if one model is stronger)
ENSEMBLE_WEIGHTS = {'A': 0.5, 'B': 0.5}

print(f'Model A : {MODEL_A}')
print(f'Model B : {MODEL_B}')
print(f'Effective batch size : {CONFIG["batch_size"]*CONFIG["grad_accum"]}')
print(f'max_len t/n/m        : {CONFIG["max_len"]}')
print(f'Epochs / patience    : {CONFIG["epochs"]} / {CONFIG["patience"]}')
print(f'Grad checkpointing   : {CONFIG["grad_ckpt"]}')
print('Est. time: ~2h (BigBird) + ~2h (Longformer) = ~4h total on single T4')


Model A : yikuan8/Clinical-Longformer
Model B : UFNLP/gatortron-base
Effective batch size : 32
max_len t/n/m        : {'t': 1024, 'n': 1024, 'm': 512}
Epochs / patience    : 10 / 3
Grad checkpointing   : True
Est. time: ~2h (BigBird) + ~2h (Longformer) = ~4h total on single T4


## 5. Preprocess & Split

In [5]:
# ── Label maps: T range corrected to 0-3 (per competition update) ────
# LABEL_MAPS = {
#     't': {0:0, 1:1, 2:2, 3:3},   # FIXED: was {1:0,2:1,3:2,4:3} — range is 0-3
#     'n': {0:0, 1:1, 2:2, 3:3},
#     'm': {0:0, 1:1},
# }
# INV_LABEL_MAPS = {task:{v:k for k,v in m.items()} for task,m in LABEL_MAPS.items()}
# CLASS_NAMES = {
#     't': ['T0','T1','T2','T3'],   # FIXED: was ['T1','T2','T3','T4']
#     'n': ['N0','N1','N2','N3'],
#     'm': ['M0','M1'],
# }
LABEL_MAPS = {
    't': {1:0, 2:1, 3:2, 4:3},   # training data is 1-4 → internal classes 0-3
    'n': {0:0, 1:1, 2:2, 3:3},
    'm': {0:0, 1:1},
}
# Submission output map: internal class 0,1,2,3 → submit as 0,1,2,3  (NOT 1,2,3,4)
INV_LABEL_MAPS = {
    't': {0:0, 1:1, 2:2, 3:3},   # ← competition correction: output 0-3
    'n': {0:0, 1:1, 2:2, 3:3},
    'm': {0:0, 1:1},
}
CLASS_NAMES = {
    't': ['T1','T2','T3','T4'],   # internal display labels (match training data)
    'n': ['N0','N1','N2','N3'],
    'm': ['M0','M1'],
}

def preprocess_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\bpT(\d)', r'pathologic tumor T\1', text)
    text = re.sub(r'\bpN(\d)', r'pathologic node N\1',  text)
    text = re.sub(r'\bpM(\d)', r'pathologic metastasis M\1', text)
    text = re.sub(r'\bpt\b',   'pathologic tumor',      text, flags=re.I)
    text = re.sub(r'\bpn\b',   'pathologic node',       text, flags=re.I)
    text = re.sub(r'\bpm\b',   'pathologic metastasis', text, flags=re.I)
    return text

print('Preprocessing text (runs once)...')
df_all['text_clean']  = df_all['text'].apply(preprocess_text)
df_sub['text_clean']  = df_sub['text'].apply(preprocess_text)
df_test['text_clean'] = df_test['text'].apply(preprocess_text)  # needed for submission
print(f'Done. {len(df_all)} train rows, {len(df_sub)} sub rows, {len(df_test)} test rows.')

def make_splits(df, task):
    df = df.dropna(subset=[task]).copy()
    df['label'] = df[task].astype(int).map(LABEL_MAPS[task])
    df = df.dropna(subset=['label'])
    df['label'] = df['label'].astype(int)
    tr, vl = train_test_split(
        df, test_size=CONFIG['val_frac'],
        stratify=df['label'], random_state=SEED
    )
    tr = tr.reset_index(drop=True)
    vl = vl.reset_index(drop=True)
    print(f'  [{task.upper()}] train={len(tr)} val={len(vl)}')
    print(f'         train dist: {tr["label"].value_counts().sort_index().to_dict()}')
    print(f'         val dist  : {vl["label"].value_counts().sort_index().to_dict()}')
    return tr, vl

print('\nSplitting:')
tr_t, vl_t = make_splits(df_all, 't')
tr_n, vl_n = make_splits(df_all, 'n')
tr_m, vl_m = make_splits(df_all, 'm')


Preprocessing text (runs once)...
Done. 6774 train rows, 2279 sub rows, 499 test rows.

Splitting:
  [T] train=4975 val=878
         train dist: {0: 1261, 1: 1687, 2: 1526, 3: 501}
         val dist  : {0: 223, 1: 298, 2: 269, 3: 88}
  [N] train=4102 val=724
         train dist: {0: 2404, 1: 1055, 2: 501, 3: 142}
         val dist  : {0: 425, 1: 186, 2: 88, 3: 25}
  [M] train=3328 val=588
         train dist: {0: 3102, 1: 226}
         val dist  : {0: 548, 1: 40}


## 6. Dataset & Collator

In [6]:
class TNMDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len, augment=False):
        self.texts   = list(texts)
        self.labels  = list(labels) if labels is not None else None
        self.tok     = tokenizer
        self.max_len = max_len
        self.aug     = augment

    def _augment(self, text):
        sents = text.split('.')
        if len(sents) > 8 and np.random.random() < 0.35:
            drop = set(np.random.choice(
                len(sents), max(1, len(sents)//10), replace=False))
            sents = [s for i,s in enumerate(sents) if i not in drop]
        return '.'.join(sents)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self._augment(self.texts[idx]) if self.aug else self.texts[idx]
        enc  = self.tok(text, max_length=self.max_len,
                        truncation=True, return_tensors='pt')
        item = {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def dynamic_pad_collate(batch):
    """Pad to true batch max-length — saves VRAM vs padding to global max_len."""
    L    = max(b['input_ids'].size(0) for b in batch)
    ids  = torch.zeros(len(batch), L, dtype=torch.long)
    mask = torch.zeros(len(batch), L, dtype=torch.long)
    for i, b in enumerate(batch):
        n = b['input_ids'].size(0)
        ids[i, :n]  = b['input_ids']
        mask[i, :n] = b['attention_mask']
    out = {'input_ids': ids, 'attention_mask': mask}
    if 'label' in batch[0]:
        out['label'] = torch.tensor([b['label'] for b in batch], dtype=torch.long)
    return out

print('Dataset + collator ready.')


Dataset + collator ready.


## 7. Model — shared architecture for both backbones

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, class_weights=None, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.cw    = class_weights
        self.ls    = label_smoothing

    def forward(self, logits, targets):
        C = logits.size(-1)
        with torch.no_grad():
            smooth = torch.full_like(logits, self.ls / (C - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        lp   = F.log_softmax(logits, -1)
        ce   = -(smooth * lp).sum(-1)
        pt   = lp.exp().gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.cw is not None:
            loss = loss * self.cw.to(logits.device)[targets]
        return loss.mean()


class TNMClassifier(nn.Module):
    """
    Works identically for Clinical-BigBird and Clinical-Longformer:
    both have hidden_size=768 and expose last_hidden_state.
    Pool = GELU( Linear([CLS ; mean-pool]) ) fed into multi-dropout head.
    """
    def __init__(self, model_name, num_labels, dropout=0.1, n_dp=5):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name, ignore_mismatched_sizes=True)
        self.backbone.gradient_checkpointing_enable()
        print(f'  Gradient checkpointing: ON  ({model_name.split("/")[-1]})')
        H = self.backbone.config.hidden_size
        self.proj     = nn.Linear(H * 2, H)
        self.dropouts = nn.ModuleList(
            [nn.Dropout(dropout + i * 0.05) for i in range(n_dp)])
        self.head     = nn.Linear(H, num_labels)
        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        out  = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        h    = out.last_hidden_state
        cls  = h[:, 0]
        mask = attention_mask.unsqueeze(-1).float()
        mean = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pool = F.gelu(self.proj(torch.cat([cls, mean], -1)))
        return sum(self.head(dp(pool)) for dp in self.dropouts) / len(self.dropouts)

print('Model class defined (shared for BigBird & Longformer).')


Model class defined (shared for BigBird & Longformer).


## 8. Training Utilities

In [8]:
def get_class_weights(labels, n):
    # Use only classes present in this split to avoid ValueError when
    # a class (e.g. T0) has zero samples in train or val.
    present = np.unique(labels)
    cw_present = compute_class_weight('balanced', classes=present, y=labels)
    # Fill weight=1.0 for any class absent from this split
    cw = np.ones(n, dtype=np.float64)
    for cls, w in zip(present, cw_present):
        cw[cls] = w
    return torch.tensor(cw, dtype=torch.float32)


def compute_metrics(preds, labels, probs, names):
    n = len(names)
    res = {
        'micro_f1': f1_score(labels, preds, average='micro'),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'micro_p' : precision_score(labels, preds, average='micro',  zero_division=0),
        'micro_r' : recall_score(labels,    preds, average='micro',  zero_division=0),
        'macro_p' : precision_score(labels, preds, average='macro',  zero_division=0),
        'macro_r' : recall_score(labels,    preds, average='macro',  zero_division=0),
    }
    try:
        res['auroc'] = (roc_auc_score(labels, probs[:, 1])
                        if n == 2
                        else roc_auc_score(labels, probs,
                                           multi_class='ovr', average='macro'))
    except Exception:
        res['auroc'] = float('nan')
    return res


def train_one_epoch(model, loader, opt, sched, crit, scaler, ga):
    model.train()
    total = 0.0
    opt.zero_grad()
    n_batches = len(loader)
    for step, batch in enumerate(loader):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbls = batch['label'].to(DEVICE)
        with autocast(enabled=CONFIG['fp16']):
            loss = crit(model(ids, mask), lbls) / ga
        scaler.scale(loss).backward()
        if (step + 1) % ga == 0 or (step + 1) == n_batches:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()
        total += loss.item() * ga
        if (step + 1) % 50 == 0:
            print(f'    batch {step+1}/{n_batches}  '
                  f'loss={total/(step+1):.4f}  '
                  f'vram={torch.cuda.memory_allocated()/1e9:.1f}GB', flush=True)
    return total / n_batches


@torch.no_grad()
def run_inference(model, loader, tta=False):
    model.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lg   = model(ids, mask)
        if tta:
            lg = (lg + model(ids, mask)) / 2
        all_logits.append(lg.cpu())
        if 'label' in batch:
            all_labels.append(batch['label'])
    logits = torch.cat(all_logits)
    probs  = F.softmax(logits, -1).numpy()
    preds  = logits.argmax(-1).numpy()
    labels = torch.cat(all_labels).numpy() if all_labels else None
    return preds, probs, labels

print('Utilities ready.')


Utilities ready.


## 9. Training Pipeline — model_tag tracks A/B checkpoints

In [9]:
def train_task(task, df_tr, df_vl, model_name, model_tag):
    """
    Train one task with one backbone.
    model_tag: 'A' or 'B' — used to namespace checkpoint files so both
               models' weights are preserved for ensemble inference.
    """
    max_len    = CONFIG['max_len'][task]
    num_labels = len(LABEL_MAPS[task])
    names      = CLASS_NAMES[task]
    ga         = CONFIG['grad_accum']

    free_vram(f'before task {task.upper()} model {model_tag}')
    t0 = time.time()

    short_name = model_name.split('/')[-1]
    print(f'\n{"="*65}')
    print(f'  Task {task.upper()} | {short_name} [{model_tag}] | '
          f'max_len={max_len} | {num_labels} classes')
    print(f'  train={len(df_tr)}  val={len(df_vl)}')
    print(f'  {vram_stats()}')
    print(f'{"="*65}')

    tok   = AutoTokenizer.from_pretrained(model_name)
    kw    = dict(batch_size=CONFIG['batch_size'],
                 collate_fn=dynamic_pad_collate,
                 num_workers=CONFIG['num_workers'],
                 pin_memory=True,
                 worker_init_fn=worker_init_fn)
    tr_ld = DataLoader(
        TNMDataset(df_tr['text_clean'], df_tr['label'], tok, max_len, augment=True),
        shuffle=True, generator=make_loader_generator(), **kw)
    vl_ld = DataLoader(
        TNMDataset(df_vl['text_clean'], df_vl['label'], tok, max_len, augment=False),
        shuffle=False, **kw)

    model = TNMClassifier(model_name, num_labels, CONFIG['dropout']).to(DEVICE)
    print(f'  Model loaded. {vram_stats()}')

    cw   = get_class_weights(df_tr['label'].values, num_labels)
    crit = FocalLoss(CONFIG['focal_gamma'], cw, CONFIG['label_smooth']).to(DEVICE)

    no_decay = ['bias', 'LayerNorm.weight']
    opt = torch.optim.AdamW([
        {'params': [p for n, p in model.backbone.named_parameters()
                    if not any(nd in n for nd in no_decay)],
         'lr': CONFIG['lr'], 'weight_decay': CONFIG['weight_decay']},
        {'params': [p for n, p in model.backbone.named_parameters()
                    if any(nd in n for nd in no_decay)],
         'lr': CONFIG['lr'], 'weight_decay': 0.0},
        {'params': list(model.proj.parameters()) + list(model.head.parameters()),
         'lr': CONFIG['lr'] * 5,
         'weight_decay': CONFIG['weight_decay']},
    ])
    total_s = (len(tr_ld) // ga) * CONFIG['epochs']
    sched   = get_cosine_schedule_with_warmup(
        opt, int(total_s * CONFIG['warmup_ratio']), total_s)
    scaler  = GradScaler(enabled=CONFIG['fp16'])

    best_f1, pat = -1.0, 0
    # ── Checkpoint filename encodes both task AND model tag ──────────────
    ckpt    = f'{OUTPUT_DIR}/best_{task}_{model_tag}.pt'
    history = {'loss': [], 'micro_f1': [], 'macro_f1': [], 'auroc': []}

    print(f'\n  Ep   Loss    Micro-F1  Macro-F1  AU-ROC   '
          f'Min/ep  TotalMin  VRAM')
    print(f'  {"-"*70}')

    for ep in range(CONFIG['epochs']):
        ep_t = time.time()

        loss = train_one_epoch(model, tr_ld, opt, sched, crit, scaler, ga)

        vp, vprob, vlab = run_inference(model, vl_ld)
        vm = compute_metrics(vp, vlab, vprob, names)

        history['loss'].append(loss)
        history['micro_f1'].append(vm['micro_f1'])
        history['macro_f1'].append(vm['macro_f1'])
        history['auroc'].append(vm['auroc'])

        ep_min  = (time.time() - ep_t)  / 60
        tot_min = (time.time() - t0)    / 60
        vram_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

        star = ' *BEST*' if vm['macro_f1'] > best_f1 else ''
        print(f'  {ep+1:2d}   {loss:.4f}  {vm["micro_f1"]:.4f}    '
              f'{vm["macro_f1"]:.4f}    {vm["auroc"]:.4f}   '
              f'{ep_min:5.1f}   {tot_min:6.1f}    {vram_gb:.1f}GB{star}',
              flush=True)

        if vm['macro_f1'] > best_f1:
            best_f1 = vm['macro_f1']
            torch.save(model.state_dict(), ckpt)
            pat = 0
        else:
            pat += 1
            if pat >= CONFIG['patience']:
                print(f'\n  Early stop at epoch {ep+1} '
                      f'(best macro_f1={best_f1:.4f})')
                break

    # ── Final eval on best checkpoint (TTA) ─────────────────────────────
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    vp, vprob, vlab = run_inference(model, vl_ld, tta=True)
    vm_best = compute_metrics(vp, vlab, vprob, names)

    tot_min = (time.time() - t0) / 60
    print(f'\n  {"─"*65}')
    print(f'  Task {task.upper()} [{model_tag}] DONE in {tot_min:.1f} min')
    print(f'  Final (TTA): '
          f'micro_f1={vm_best["micro_f1"]:.4f}  '
          f'macro_f1={vm_best["macro_f1"]:.4f}  '
          f'auroc={vm_best["auroc"]:.4f}')
    print()
    print(classification_report(vlab, vp, target_names=names))

    saved_probs  = vprob.copy()
    saved_preds  = vp.copy()
    saved_labels = vlab.copy()

    del opt, sched, scaler, crit
    model.cpu()
    del model
    free_vram(f'after task {task.upper()} model {model_tag}')

    return tok, vm_best, saved_preds, saved_probs, saved_labels, history

print('train_task() ready.')


train_task() ready.


## 10. Train Model A  — T, N, M

In [ ]:
# # ── MODEL A training is SKIPPED — Only GatorTron (Model B) is trained ──
# # Set TRAIN_MODEL_A = True here if you want to run the full ensemble later.
# TRAIN_MODEL_A = False

# if TRAIN_MODEL_A:
#     print('=' * 65)
#     print('  TRAINING MODEL A: Clinical-Longformer')
#     print('=' * 65)
#     tok_t_A, mt_t_A, pred_t_A, prob_t_A, lab_t, hist_t_A = train_task(
#         't', tr_t, vl_t, MODEL_A, 'A')
# else:
#     print('>>> Model A (Longformer) skipped — TRAIN_MODEL_A=False')
#     print('    Ensemble cells (38-49) will not be available.')
#     # Provide placeholder labels so downstream split-dependent checks still pass
#     import numpy as np
#     lab_t = vl_t['label'].values
#     lab_n = vl_n['label'].values
#     lab_m = vl_m['label'].values


In [ ]:
# if TRAIN_MODEL_A:
#     tok_n_A, mt_n_A, pred_n_A, prob_n_A, lab_n, hist_n_A = train_task(
#         'n', tr_n, vl_n, MODEL_A, 'A')
#     tok_m_A, mt_m_A, pred_m_A, prob_m_A, lab_m, hist_m_A = train_task(
#         'm', tr_m, vl_m, MODEL_A, 'A')
#     print('\n>>> Model A training complete.')
# else:
#     print('>>> Model A N/M tasks skipped.')


## Model A Result Summary

In [ ]:
# # ── Model A (BigBird) Result Summary ──────────────────────────────────────

# # Use Model A metrics and probabilities
# totals = len(vl_t) + len(vl_n) + len(vl_m)

# def wavg(k):
#     return (mt_t_A[k]*len(vl_t) + mt_n_A[k]*len(vl_n) + mt_m_A[k]*len(vl_m)) / totals

# rows = []
# for task, mt in [('T14', mt_t_A), ('N03', mt_n_A), ('M01', mt_m_A)]:
#     rows.append({'Task': task, **{k: round(v,4) for k,v in mt.items()}})
# rows.append({'Task':'OVERALL', **{k: round(wavg(k),4) for k in mt_t_A}})
# print(pd.DataFrame(rows).to_string(index=False))

# print('\n=== vs Leaderboard (Model A — BigBird) ===')
# board = pd.DataFrame([
#     {'Model':'BigBird [A]',  'Micro-F1':round(wavg('micro_f1'),4),
#      'Macro-F1':round(wavg('macro_f1'),4), 'AU-ROC':round(wavg('auroc'),4)},
#     {'Model':'#1 abebe9849','Micro-F1':0.90,'Macro-F1':0.81,'AU-ROC':'N/A'},
#     {'Model':'#2 madrueno', 'Micro-F1':0.84,'Macro-F1':0.70,'AU-ROC':'N/A'},
# ])
# print(board.to_string(index=False))
# gap = round(wavg('micro_f1'), 4) - 0.90
# print(f'\nMicro-F1 delta vs #1: {gap:+.4f}')

# # ── M threshold tuning on Model A probabilities ───────────────────────────
# best_thresh_A, best_mf1_A = 0.5, 0.0
# for t in np.arange(0.10, 0.70, 0.02):
#     p = (prob_m_A[:, 1] > t).astype(int)
#     f = f1_score(lab_m, p, average='macro', zero_division=0)
#     if f > best_mf1_A:
#         best_mf1_A, best_thresh_A = f, t

# pred_m_tuned_A = (prob_m_A[:, 1] > best_thresh_A).astype(int)
# mt_m_tuned_A   = compute_metrics(pred_m_tuned_A, lab_m, prob_m_A, CLASS_NAMES['m'])
# print(f'\nBest threshold  : {best_thresh_A:.2f}')
# print(f'macro_f1 before : {mt_m_A["macro_f1"]:.4f}')
# print(f'macro_f1 after  : {mt_m_tuned_A["macro_f1"]:.4f}')
# print(f'micro_f1 after  : {mt_m_tuned_A["micro_f1"]:.4f}')
# print()
# print(classification_report(lab_m, pred_m_tuned_A, target_names=CLASS_NAMES['m']))

## Training Curves — Model A 

In [ ]:
# # ── Cell 1: Training Curves — Model A (BigBird) ───────────────────────────

# fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# for ax, (hist, task) in zip(axes,
#         [(hist_t_A,'T'),(hist_n_A,'N'),(hist_m_A,'M')]):
#     ep  = range(1, len(hist['loss'])+1)
#     ax2 = ax.twinx()
#     ax.plot(ep,  hist['loss'],     'b-o', ms=5, label='Loss')
#     ax2.plot(ep, hist['macro_f1'], 'r-s', ms=5, label='Macro-F1')
#     ax2.plot(ep, hist['auroc'],    'g-^', ms=5, label='AU-ROC')
#     ax.set_title(f'Task {task} — BigBird [A]')
#     ax.set_xlabel('Epoch')
#     ax.set_ylabel('Loss', color='b')
#     ax2.set_ylabel('Score', color='r')
#     ax.legend(loc='upper left', fontsize=8)
#     ax2.legend(loc='upper right', fontsize=8)
# plt.tight_layout()
# plt.savefig(f'{OUTPUT_DIR}/curves_A.png', dpi=100)
# plt.show()

## Confusion Matrices — Model A 

In [ ]:
# # ── Cell 2: Confusion Matrices — Model A (BigBird) ────────────────────────

# fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# for ax, (p, l, names, task) in zip(axes, [
#     (pred_t_A,       lab_t, CLASS_NAMES['t'], 'T14 [BigBird A]'),
#     (pred_n_A,       lab_n, CLASS_NAMES['n'], 'N03 [BigBird A]'),
#     (pred_m_tuned_A, lab_m, CLASS_NAMES['m'], 'M01 [BigBird A]'),
# ]):
#     cm  = confusion_matrix(l, p)
#     pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
#     ann = np.array([[f'{cm[i,j]}\n({pct[i,j]:.0f}%)'
#                      for j in range(cm.shape[1])]
#                     for i in range(cm.shape[0])])
#     sns.heatmap(cm, annot=ann, fmt='', ax=ax, cmap='Blues',
#                 xticklabels=names, yticklabels=names)
#     ax.set_title(task)
#     ax.set_xlabel('Predicted')
#     ax.set_ylabel('True')
# plt.tight_layout()
# plt.savefig(f'{OUTPUT_DIR}/conf_mat_A.png', dpi=100)
# plt.show()

## Submission CSV — Model A only

In [ ]:
# # ── Cell 3: Submission CSV — Model A (BigBird) only ───────────────────────

# def predict_submission(df_sub, task, tok, max_len, model_name, model_tag,
#                        threshold=0.5):
#     """Load best checkpoint for model_tag, predict, then free GPU."""
#     ckpt       = f'{OUTPUT_DIR}/best_{task}_{model_tag}.pt'
#     num_labels = len(LABEL_MAPS[task])

#     model = TNMClassifier(model_name, num_labels,
#                           CONFIG['dropout']).to(DEVICE)
#     model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

#     ds = TNMDataset(df_sub['text_clean'], None, tok, max_len)
#     ld = DataLoader(ds, batch_size=CONFIG['batch_size'],
#                     collate_fn=dynamic_pad_collate,
#                     num_workers=CONFIG['num_workers'],
#                     pin_memory=True,
#                  worker_init_fn=worker_init_fn)

#     preds, probs, _ = run_inference(model, ld, tta=True)

#     if task == 'm' and threshold != 0.5:
#         preds = (probs[:, 1] > threshold).astype(int)

#     model.cpu()
#     del model
#     free_vram(f'after {task.upper()} [{model_tag}] inference')

#     return [INV_LABEL_MAPS[task][p] for p in preds]


# print('Running submission inference — Model A (BigBird)...')
# print('  T...', flush=True)
# t_preds = predict_submission(df_sub, 't', tok_t_A,
#                              CONFIG['max_len']['t'], MODEL_A, 'A')
# print('  N...', flush=True)
# n_preds = predict_submission(df_sub, 'n', tok_n_A,
#                              CONFIG['max_len']['n'], MODEL_A, 'A')
# print('  M...', flush=True)
# m_preds = predict_submission(df_sub, 'm', tok_m_A,
#                              CONFIG['max_len']['m'], MODEL_A, 'A',
#                              best_thresh_A)

# submission = pd.DataFrame({
#     'patient_filename': df_sub['patient_filename'],
#     't': t_preds,
#     'n': n_preds,
#     'm': m_preds,
# })
# submission.to_csv(f'{OUTPUT_DIR}/submission_A.csv', index=False)
# print(f'\nsubmission_A.csv saved — {len(submission)} rows')
# print(f'  T: {submission["t"].value_counts().sort_index().to_dict()}')
# print(f'  N: {submission["n"].value_counts().sort_index().to_dict()}')
# print(f'  M: {submission["m"].value_counts().sort_index().to_dict()}')
# submission.head(10)

## 11. Train Model B (GatorTron-Base) — T, N, M

In [10]:
# ── MODEL B: GatorTron-Base — Train or Restore from Checkpoint ──────────
import logging, os
logging.getLogger('transformers').setLevel(logging.ERROR)  # silence padding warnings

_max_len_B = {'t': 512, 'n': 512, 'm': 256}

# ── Checkpoint paths ──────────────────────────────────────────────────────
_ckpt_t_B = f'{OUTPUT_DIR}/best_t_B.pt'
_ckpt_n_B = f'{OUTPUT_DIR}/best_n_B.pt'
_ckpt_m_B = f'{OUTPUT_DIR}/best_m_B.pt'
_all_ckpts_exist = all(os.path.isfile(p) for p in [_ckpt_t_B, _ckpt_n_B, _ckpt_m_B])

# ── Helper: load a saved checkpoint and run validation inference ──────────
def _restore_B(task, df_vl, max_len):
    """Reload best_<task>_B.pt, run TTA inference on val set.
    Returns the same (tok, mt, preds, probs, labels, hist) tuple
    as train_task() so all downstream cells work unchanged.
    """
    ckpt       = f'{OUTPUT_DIR}/best_{task}_B.pt'
    num_labels = len(LABEL_MAPS[task])
    names      = CLASS_NAMES[task]
    tok   = AutoTokenizer.from_pretrained(MODEL_B)
    ds    = TNMDataset(df_vl['text_clean'], df_vl['label'], tok, max_len)
    ld    = DataLoader(ds, batch_size=CONFIG['batch_size'],
                       collate_fn=dynamic_pad_collate,
                       num_workers=CONFIG['num_workers'],
                       pin_memory=True,
                       worker_init_fn=worker_init_fn)
    model = TNMClassifier(MODEL_B, num_labels, CONFIG['dropout']).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    preds, probs, labels = run_inference(model, ld, tta=True)
    mt    = compute_metrics(preds, labels, probs, names)
    # Empty history stubs — training curves won't plot, but no crash
    hist  = {'loss': [], 'micro_f1': [], 'macro_f1': [], 'auroc': []}
    model.cpu(); del model
    free_vram(f'restored {task.upper()} [B]')
    print(f'  [B] Task {task.upper()}  '
          f'micro_f1={mt["micro_f1"]:.4f}  macro_f1={mt["macro_f1"]:.4f}')
    return tok, mt, preds, probs, labels, hist


if _all_ckpts_exist:
    # ── FAST PATH: checkpoints already saved — skip training entirely ────
    print('=' * 65)
    print('  RESTORING MODEL B: GatorTron-Base  (checkpoints found)')
    print('=' * 65)
    for p in [_ckpt_t_B, _ckpt_n_B, _ckpt_m_B]:
        size_mb = os.path.getsize(p) / 1e6
        print(f'  ✔  {p}  ({size_mb:.0f} MB)')
    print()

    tok_t_B, mt_t_B, pred_t_B, prob_t_B, _, hist_t_B = _restore_B('t', vl_t, _max_len_B['t'])
    tok_n_B, mt_n_B, pred_n_B, prob_n_B, _, hist_n_B = _restore_B('n', vl_n, _max_len_B['n'])
    tok_m_B, mt_m_B, pred_m_B, prob_m_B, _, hist_m_B = _restore_B('m', vl_m, _max_len_B['m'])

    print('\n>>> Model B restored from checkpoints — no retraining needed.')

else:
    # ── TRAIN PATH: one or more checkpoints missing — train from scratch ──
    _missing = [p for p in [_ckpt_t_B, _ckpt_n_B, _ckpt_m_B] if not os.path.isfile(p)]
    print('=' * 65)
    print('  TRAINING MODEL B: GatorTron-Base')
    print('=' * 65)
    print(f'  Missing checkpoints: {_missing}')
    print()

    # GatorTron-base hard limit is 512 tokens — override max_len for Model B
    _max_len_backup = CONFIG['max_len'].copy()
    CONFIG['max_len'] = _max_len_B

    # NOTE: lab_t, lab_n, lab_m are reused from Model A placeholders / Cell 20
    CONFIG['epochs'] = 10
    tok_t_B, mt_t_B, pred_t_B, prob_t_B, _, hist_t_B = train_task(
        't', tr_t, vl_t, MODEL_B, 'B')
    CONFIG['epochs'] = 10
    tok_n_B, mt_n_B, pred_n_B, prob_n_B, _, hist_n_B = train_task(
        'n', tr_n, vl_n, MODEL_B, 'B')
    CONFIG['epochs'] = 10
    tok_m_B, mt_m_B, pred_m_B, prob_m_B, _, hist_m_B = train_task(
        'm', tr_m, vl_m, MODEL_B, 'B')

    # Restore original max_len for any subsequent cells
    CONFIG['max_len'] = _max_len_backup

    print('\n>>> Model B training complete.')

# # ── Sanity check ──────────────────────────────────────────────────────────
# assert len(lab_t) == len(prob_t_B), 'T label/prob length mismatch'
# assert len(lab_n) == len(prob_n_B), 'N label/prob length mismatch'
# assert len(lab_m) == len(prob_m_B), 'M label/prob length mismatch'
# print('Label consistency check passed.')


  TRAINING MODEL B: GatorTron-Base
  Missing checkpoints: ['/kaggle/working/best_t_B.pt', '/kaggle/working/best_n_B.pt', '/kaggle/working/best_m_B.pt']

  [before task T model B] VRAM after free -> alloc=0.00GB  reserved=0.00GB  free=15.64GB/15.6GB

  Task T | gatortron-base [B] | max_len=512 | 4 classes
  train=4975  val=878
  alloc=0.00GB  reserved=0.00GB  free=15.64GB/15.6GB


config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/713M [00:00<?, ?B/s]

  Gradient checkpointing: ON  (gatortron-base)
  Model loaded. alloc=1.43GB  reserved=1.43GB  free=14.20GB/15.6GB

  Ep   Loss    Micro-F1  Macro-F1  AU-ROC   Min/ep  TotalMin  VRAM
  ----------------------------------------------------------------------
    batch 50/1244  loss=0.8359  vram=5.7GB
    batch 100/1244  loss=0.8016  vram=5.7GB
    batch 150/1244  loss=0.7909  vram=5.7GB
    batch 200/1244  loss=0.7914  vram=4.3GB
    batch 250/1244  loss=0.7885  vram=5.7GB
    batch 300/1244  loss=0.7877  vram=5.7GB
    batch 350/1244  loss=0.7726  vram=5.7GB
    batch 400/1244  loss=0.7595  vram=4.3GB
    batch 450/1244  loss=0.7494  vram=5.7GB
    batch 500/1244  loss=0.7392  vram=5.7GB
    batch 550/1244  loss=0.7313  vram=5.7GB
    batch 600/1244  loss=0.7213  vram=4.3GB
    batch 650/1244  loss=0.7145  vram=5.7GB
    batch 700/1244  loss=0.7071  vram=5.7GB
    batch 750/1244  loss=0.7013  vram=5.7GB
    batch 800/1244  loss=0.6916  vram=4.3GB
    batch 850/1244  loss=0.6871  vram=5.7G

## Model B (GatorTron-Base) Result Summary

In [11]:
# ── Model B (GatorTron-Base) Result Summary ────────────────────────────────────

totals = len(vl_t) + len(vl_n) + len(vl_m)

def wavg_B(k):
    return (mt_t_B[k]*len(vl_t) + mt_n_B[k]*len(vl_n) + mt_m_B[k]*len(vl_m)) / totals

rows = []
for task, mt in [('T14', mt_t_B), ('N03', mt_n_B), ('M01', mt_m_B)]:
    rows.append({'Task': task, **{k: round(v, 4) for k, v in mt.items()}})
rows.append({'Task': 'OVERALL', **{k: round(wavg_B(k), 4) for k in mt_t_B}})
print(pd.DataFrame(rows).to_string(index=False))

print('\n=== vs Leaderboard (Model B — GatorTron-Base) ===')
board = pd.DataFrame([
    {'Model': 'GatorTron-Base [B]', 'Micro-F1': round(wavg_B('micro_f1'), 4),
     'Macro-F1': round(wavg_B('macro_f1'), 4), 'AU-ROC': round(wavg_B('auroc'), 4)},
    {'Model': '#1 abebe9849',       'Micro-F1': 0.90, 'Macro-F1': 0.81, 'AU-ROC': 'N/A'},
    {'Model': '#2 madrueno',        'Micro-F1': 0.84, 'Macro-F1': 0.70, 'AU-ROC': 'N/A'},
])
print(board.to_string(index=False))
gap = round(wavg_B('micro_f1'), 4) - 0.90
print(f'\nMicro-F1 delta vs #1: {gap:+.4f}')

# ── M threshold tuning on Model B probabilities ───────────────────────────
best_thresh_B, best_mf1_B = 0.5, 0.0
for t in np.arange(0.10, 0.70, 0.02):
    p = (prob_m_B[:, 1] > t).astype(int)
    f = f1_score(lab_m, p, average='macro', zero_division=0)
    if f > best_mf1_B:
        best_mf1_B, best_thresh_B = f, t

pred_m_tuned_B = (prob_m_B[:, 1] > best_thresh_B).astype(int)
mt_m_tuned_B   = compute_metrics(pred_m_tuned_B, lab_m, prob_m_B, CLASS_NAMES['m'])
print(f'\nBest threshold  : {best_thresh_B:.2f}')
print(f'macro_f1 before : {mt_m_B["macro_f1"]:.4f}')
print(f'macro_f1 after  : {mt_m_tuned_B["macro_f1"]:.4f}')
print(f'micro_f1 after  : {mt_m_tuned_B["micro_f1"]:.4f}')
print()
print(classification_report(lab_m, pred_m_tuned_B, target_names=CLASS_NAMES['m']))

NameError: name 'best_thresh_B' is not defined

In [ ]:
# # ── Model B (GatorTron-Base) Result Summary ────────────────────────────────────

# totals = len(vl_t) + len(vl_n) + len(vl_m)

# # Model A weighted average (needed for comparison table)
# def wavg(k):
#     return (mt_t_A[k]*len(vl_t) + mt_n_A[k]*len(vl_n) + mt_m_A[k]*len(vl_m)) / totals

# # Model B weighted average
# def wavg_B(k):
#     return (mt_t_B[k]*len(vl_t) + mt_n_B[k]*len(vl_n) + mt_m_B[k]*len(vl_m)) / totals

# rows = []
# for task, mt in [('T14', mt_t_B), ('N03', mt_n_B), ('M01', mt_m_B)]:
#     rows.append({'Task': task, **{k: round(v,4) for k,v in mt.items()}})
# rows.append({'Task':'OVERALL', **{k: round(wavg_B(k),4) for k in mt_t_B}})
# print(pd.DataFrame(rows).to_string(index=False))

# print('\n=== vs Leaderboard (Model B — Longformer) ===')
# board = pd.DataFrame([
#     {'Model':'Longformer [B]', 'Micro-F1':round(wavg_B('micro_f1'),4),
#      'Macro-F1':round(wavg_B('macro_f1'),4), 'AU-ROC':round(wavg_B('auroc'),4)},
#     {'Model':'BigBird [A]',    'Micro-F1':round(wavg('micro_f1'),4),
#      'Macro-F1':round(wavg('macro_f1'),4),   'AU-ROC':round(wavg('auroc'),4)},
#     {'Model':'#1 abebe9849',   'Micro-F1':0.90,'Macro-F1':0.81,'AU-ROC':'N/A'},
#     {'Model':'#2 madrueno',    'Micro-F1':0.84,'Macro-F1':0.70,'AU-ROC':'N/A'},
# ])
# print(board.to_string(index=False))
# gap = round(wavg_B('micro_f1'), 4) - 0.90
# print(f'\nMicro-F1 delta vs #1: {gap:+.4f}')

# # ── M threshold tuning on Model B probabilities ───────────────────────────
# best_thresh_B, best_mf1_B = 0.5, 0.0
# for t in np.arange(0.10, 0.70, 0.02):
#     p = (prob_m_B[:, 1] > t).astype(int)
#     f = f1_score(lab_m, p, average='macro', zero_division=0)
#     if f > best_mf1_B:
#         best_mf1_B, best_thresh_B = f, t

# pred_m_tuned_B = (prob_m_B[:, 1] > best_thresh_B).astype(int)
# mt_m_tuned_B   = compute_metrics(pred_m_tuned_B, lab_m, prob_m_B, CLASS_NAMES['m'])
# print(f'\nBest threshold  : {best_thresh_B:.2f}')
# print(f'macro_f1 before : {mt_m_B["macro_f1"]:.4f}')
# print(f'macro_f1 after  : {mt_m_tuned_B["macro_f1"]:.4f}')
# print(f'micro_f1 after  : {mt_m_tuned_B["micro_f1"]:.4f}')
# print()
# print(classification_report(lab_m, pred_m_tuned_B, target_names=CLASS_NAMES['m']))

## Submission CSV — Model B (GatorTron-Base) only

In [14]:
# ── Submission CSV — Model B (GatorTron-Base) only ────────────────────────

def predict_submission(df_test, task, tok, max_len, model_name, model_tag,
                       threshold=0.5):
    """Load best checkpoint for model_tag, predict, then free GPU."""
    ckpt       = f'{OUTPUT_DIR}/best_{task}_{model_tag}.pt'
    num_labels = len(LABEL_MAPS[task])

    model = TNMClassifier(model_name, num_labels,
                          CONFIG['dropout']).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    ds = TNMDataset(df_test['text_clean'], None, tok, max_len)
    ld = DataLoader(ds, batch_size=CONFIG['batch_size'],
                    collate_fn=dynamic_pad_collate,
                    num_workers=CONFIG['num_workers'],
                    pin_memory=True,
                    worker_init_fn=worker_init_fn)

    preds, probs, _ = run_inference(model, ld, tta=True)

    if task == 'm' and threshold != 0.5:
        preds = (probs[:, 1] > threshold).astype(int)

    model.cpu()
    del model
    free_vram(f'after {task.upper()} [{model_tag}] inference')

    return [INV_LABEL_MAPS[task][p] for p in preds]


# GatorTron-Base hard limit: 512 tokens
_max_len_B = {'t': 512, 'n': 512, 'm': 256}

print('Running submission inference — Model B (GatorTron-Base)...')
print('  T...', flush=True)
t_preds = predict_submission(df_test, 't', tok_t_B,
                             _max_len_B['t'], MODEL_B, 'B')
print('  N...', flush=True)
n_preds = predict_submission(df_test, 'n', tok_n_B,
                             _max_len_B['n'], MODEL_B, 'B')
print('  M...', flush=True)
m_preds = predict_submission(df_test, 'm', tok_m_B,
                             _max_len_B['m'], MODEL_B, 'B')   # tuned threshold from Result Summary cell

submission_B = pd.DataFrame({
    'patient_filename': df_test['id'],  # corrected here
    't': t_preds,
    'n': n_preds,
    'm': m_preds,
})
submission_B.to_csv(f'{OUTPUT_DIR}/submission_Gatotron_test_hard.csv', index=False)
print(f'\nsubmission_Gatotron_test_hard.csv saved — {len(submission_B)} rows')
print(f'  T: {submission_B["t"].value_counts().sort_index().to_dict()}')
print(f'  N: {submission_B["n"].value_counts().sort_index().to_dict()}')
print(f'  M: {submission_B["m"].value_counts().sort_index().to_dict()}')
submission_B.head(10)

Running submission inference — Model B (GatorTron-Base)...
  T...
  Gradient checkpointing: ON  (gatortron-base)
  [after T [B] inference] VRAM after free -> alloc=0.02GB  reserved=0.09GB  free=15.55GB/15.6GB
  N...
  Gradient checkpointing: ON  (gatortron-base)
  [after N [B] inference] VRAM after free -> alloc=0.02GB  reserved=0.09GB  free=15.55GB/15.6GB
  M...
  Gradient checkpointing: ON  (gatortron-base)
  [after M [B] inference] VRAM after free -> alloc=0.02GB  reserved=0.09GB  free=15.55GB/15.6GB

submission_Gatotron_test_hard.csv saved — 499 rows
  T: {0: 104, 1: 127, 2: 259, 3: 9}
  N: {0: 290, 1: 124, 2: 71, 3: 14}
  M: {0: 410, 1: 89}


,patient_filename,t,n,m
0,TNM_0000,0,0,0
1,TNM_0001,2,1,0
2,TNM_0002,2,0,0
3,TNM_0003,0,0,0
4,TNM_0004,2,1,0
5,TNM_0005,2,0,0
6,TNM_0006,2,1,0
7,TNM_0007,2,2,1
8,TNM_0008,2,0,0
9,TNM_0009,1,1,1


## 12. Ensemble — Average Probabilities on Validation Set

In [ ]:
# # ── Task-specific ensemble weights based on Macro-F1 performance ──────────

# # Task T: Longformer stronger (0.7811 vs 0.7677)
# wA_t = 0.7677 / (0.7677 + 0.7811)
# wB_t = 0.7811 / (0.7677 + 0.7811)

# # Task N: BigBird MUCH stronger (0.8222 vs 0.6303)
# wA_n = 0.8222 / (0.8222 + 0.6303)
# wB_n = 0.6303 / (0.8222 + 0.6303)

# # Task M: Longformer stronger (0.6566 vs 0.6371)
# wA_m = 0.6371 / (0.6371 + 0.6566)
# wB_m = 0.6566 / (0.6371 + 0.6566)

# print(f'Task T weights: A={wA_t:.3f}  B={wB_t:.3f}')
# print(f'Task N weights: A={wA_n:.3f}  B={wB_n:.3f}')
# print(f'Task M weights: A={wA_m:.3f}  B={wB_m:.3f}')

# # ── Weighted average of softmax probabilities ──────────────────────────────
# prob_t_ens = wA_t * prob_t_A + wB_t * prob_t_B
# prob_n_ens = wA_n * prob_n_A + wB_n * prob_n_B
# prob_m_ens = wA_m * prob_m_A + wB_m * prob_m_B

# pred_t_ens = prob_t_ens.argmax(-1)
# pred_n_ens = prob_n_ens.argmax(-1)
# pred_m_ens = prob_m_ens.argmax(-1)  # threshold tuned later

# mt_t_ens = compute_metrics(pred_t_ens, lab_t, prob_t_ens, CLASS_NAMES['t'])
# mt_n_ens = compute_metrics(pred_n_ens, lab_n, prob_n_ens, CLASS_NAMES['n'])
# mt_m_ens = compute_metrics(pred_m_ens, lab_m, prob_m_ens, CLASS_NAMES['m'])

# # ── Comparison table ───────────────────────────────────────────────────────
# totals = len(vl_t) + len(vl_n) + len(vl_m)

# def wavg_(k, a, b, c):
#     return (a[k]*len(vl_t) + b[k]*len(vl_n) + c[k]*len(vl_m)) / totals

# print('\nValidation metrics — individual vs ensemble')
# print(f'  {"Model":<12}  {"Micro-F1":>9}  {"Macro-F1":>9}  {"AUROC":>8}')
# print(f'  {"-"*45}')
# for tag, mt_t_, mt_n_, mt_m_ in [
#     ("BigBird",    mt_t_A, mt_n_A, mt_m_A),
#     ("Longformer", mt_t_B, mt_n_B, mt_m_B),
#     ("ENSEMBLE",   mt_t_ens, mt_n_ens, mt_m_ens),
# ]:
#     print(f'  {tag:<12}  '
#           f'{wavg_("micro_f1", mt_t_, mt_n_, mt_m_):>9.4f}  '
#           f'{wavg_("macro_f1", mt_t_, mt_n_, mt_m_):>9.4f}  '
#           f'{wavg_("auroc",    mt_t_, mt_n_, mt_m_):>8.4f}')

## Submission CSV Ensemble

In [ ]:
# # ── Generate Submission CSV — Task-Specific Weighted Ensemble ──────────────

# def predict_submission_single(df_sub, task, tok, max_len, model_name, model_tag):
#     """Load best checkpoint, run inference, free GPU. Returns probabilities."""
#     ckpt       = f'{OUTPUT_DIR}/best_{task}_{model_tag}.pt'
#     num_labels = len(LABEL_MAPS[task])

#     model = TNMClassifier(model_name, num_labels, CONFIG['dropout']).to(DEVICE)
#     model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

#     ds = TNMDataset(df_sub['text_clean'], None, tok, max_len)
#     ld = DataLoader(ds, batch_size=CONFIG['batch_size'],
#                     collate_fn=dynamic_pad_collate,
#                     num_workers=CONFIG['num_workers'],
#                     pin_memory=True,
#                  worker_init_fn=worker_init_fn)

#     _, probs, _ = run_inference(model, ld, tta=True)

#     model.cpu()
#     del model
#     free_vram(f'after {task.upper()} [{model_tag}] inference')
#     return probs

# print('Running submission ensemble inference...')

# # ── Model A inference ──────────────────────────────────────────────────────
# print('\n--- Model A (BigBird) ---')
# sub_prob_t_A = predict_submission_single(
#     df_sub, 't', tok_t_A, CONFIG['max_len']['t'], MODEL_A, 'A')
# sub_prob_n_A = predict_submission_single(
#     df_sub, 'n', tok_n_A, CONFIG['max_len']['n'], MODEL_A, 'A')
# sub_prob_m_A = predict_submission_single(
#     df_sub, 'm', tok_m_A, CONFIG['max_len']['m'], MODEL_A, 'A')

# # ── Model B inference ──────────────────────────────────────────────────────
# print('\n--- Model B (Longformer) ---')
# sub_prob_t_B = predict_submission_single(
#     df_sub, 't', tok_t_B, CONFIG['max_len']['t'], MODEL_B, 'B')
# sub_prob_n_B = predict_submission_single(
#     df_sub, 'n', tok_n_B, CONFIG['max_len']['n'], MODEL_B, 'B')
# sub_prob_m_B = predict_submission_single(
#     df_sub, 'm', tok_m_B, CONFIG['max_len']['m'], MODEL_B, 'B')

# # ── Task-specific weighted ensemble ───────────────────────────────────────
# sub_prob_t_ens = wA_t * sub_prob_t_A + wB_t * sub_prob_t_B
# sub_prob_n_ens = wA_n * sub_prob_n_A + wB_n * sub_prob_n_B
# sub_prob_m_ens = wA_m * sub_prob_m_A + wB_m * sub_prob_m_B

# # ── Final predictions ──────────────────────────────────────────────────────
# t_preds = [INV_LABEL_MAPS['t'][p] for p in sub_prob_t_ens.argmax(-1)]
# n_preds = [INV_LABEL_MAPS['n'][p] for p in sub_prob_n_ens.argmax(-1)]

# # M: tune threshold on ensemble validation probabilities
# best_thresh_ens, best_mf1_ens = 0.5, 0.0
# for t in np.arange(0.10, 0.70, 0.02):
#     p = (prob_m_ens[:, 1] > t).astype(int)
#     f = f1_score(lab_m, p, average='macro', zero_division=0)
#     if f > best_mf1_ens:
#         best_mf1_ens, best_thresh_ens = f, t

# print(f'\nEnsemble M threshold: {best_thresh_ens:.2f}')
# m_raw   = (sub_prob_m_ens[:, 1] > best_thresh_ens).astype(int)
# m_preds = [INV_LABEL_MAPS['m'][p] for p in m_raw]

# # ── Save submission ────────────────────────────────────────────────────────
# submission = pd.DataFrame({
#     'patient_filename': df_sub['patient_filename'],
#     't': t_preds,
#     'n': n_preds,
#     'm': m_preds,
# })
# submission.to_csv(f'{OUTPUT_DIR}/submission_ensemble.csv', index=False)
# print(f'\nsubmission_ensemble.csv saved — {len(submission)} rows')
# print(f'  T: {submission["t"].value_counts().sort_index().to_dict()}')
# print(f'  N: {submission["n"].value_counts().sort_index().to_dict()}')
# print(f'  M: {submission["m"].value_counts().sort_index().to_dict()}')
# submission.head(10)

## 13. Results Summary — per-task breakdown

In [ ]:
# totals = len(vl_t) + len(vl_n) + len(vl_m)

# def wavg(k, mt_t_, mt_n_, mt_m_):
#     return (mt_t_[k]*len(vl_t) + mt_n_[k]*len(vl_n) + mt_m_[k]*len(vl_m)) / totals

# rows = []
# for label, mt_t_, mt_n_, mt_m_ in [
#     ('BigBird-T14',    mt_t_A, mt_n_A, mt_m_A),
#     ('Longformer-T14', mt_t_B, mt_n_B, mt_m_B),
#     ('Ensemble-T14',   mt_t_ens, mt_n_ens, mt_m_ens),
# ]:
#     rows.append({'Model/Task': label,
#                  **{k: round(v,4) for k,v in mt_t_.items()}})

# print(pd.DataFrame(rows).to_string(index=False))

# print('\n=== vs Leaderboard ===')
# board = pd.DataFrame([
#     {'Model':'Ensemble',    'Micro-F1':round(wavg('micro_f1', mt_t_ens, mt_n_ens, mt_m_ens),4),
#      'Macro-F1':round(wavg('macro_f1', mt_t_ens, mt_n_ens, mt_m_ens),4),
#      'AU-ROC':round(wavg('auroc', mt_t_ens, mt_n_ens, mt_m_ens),4)},
#     {'Model':'#1 abebe9849','Micro-F1':0.90,'Macro-F1':0.81,'AU-ROC':'N/A'},
#     {'Model':'#2 madrueno', 'Micro-F1':0.84,'Macro-F1':0.70,'AU-ROC':'N/A'},
# ])
# print(board.to_string(index=False))
# gap = round(wavg('micro_f1', mt_t_ens, mt_n_ens, mt_m_ens), 4) - 0.90
# print(f'\nMicro-F1 delta vs #1: {gap:+.4f}')


## 14. M01 Threshold Tuning on Ensemble Probabilities

In [ ]:
# best_thresh, best_mf1 = 0.5, 0.0
# for t in np.arange(0.10, 0.70, 0.02):
#     p = (prob_m_ens[:, 1] > t).astype(int)
#     f = f1_score(lab_m, p, average='macro', zero_division=0)
#     if f > best_mf1:
#         best_mf1, best_thresh = f, t

# pred_m_tuned = (prob_m_ens[:, 1] > best_thresh).astype(int)
# mt_m_tuned   = compute_metrics(pred_m_tuned, lab_m, prob_m_ens, CLASS_NAMES['m'])

# print(f'Best threshold  : {best_thresh:.2f}')
# print(f'macro_f1 before : {mt_m_ens["macro_f1"]:.4f}')
# print(f'macro_f1 after  : {mt_m_tuned["macro_f1"]:.4f}')
# print(f'micro_f1 after  : {mt_m_tuned["micro_f1"]:.4f}')
# print()
# print(classification_report(lab_m, pred_m_tuned, target_names=CLASS_NAMES['m']))


## 15. Plots — Training Curves & Confusion Matrices

In [ ]:
# # ── Training curves for both models ──────────────────────────────────────
# fig, axes = plt.subplots(2, 3, figsize=(16, 8))
# for row, (tag, hists) in enumerate([
#     # ('BigBird',    [hist_t_A, hist_n_A, hist_m_A]),
#     ('Gatotron', [hist_t_B, hist_n_B, hist_m_B]),
# ]):
#     for col, (hist, task) in enumerate(zip(hists, ['T','N','M'])):
#         ax  = axes[row, col]
#         ep  = range(1, len(hist['loss'])+1)
#         ax2 = ax.twinx()
#         ax.plot(ep,  hist['loss'],     'b-o', ms=4, label='Loss')
#         ax2.plot(ep, hist['macro_f1'], 'r-s', ms=4, label='Macro-F1')
#         ax2.plot(ep, hist['auroc'],    'g-^', ms=4, label='AU-ROC')
#         ax.set_title(f'{tag} — Task {task}')
#         ax.set_xlabel('Epoch')
#         ax.set_ylabel('Loss', color='b')
#         ax2.set_ylabel('Score', color='r')
#         ax.legend(loc='upper left',  fontsize=7)
#         ax2.legend(loc='upper right', fontsize=7)
# plt.tight_layout()
# plt.savefig(f'{OUTPUT_DIR}/curves.png', dpi=100)
# plt.show()


In [ ]:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# # Compute confusion matrix for tuned Model B predictions
# cm = confusion_matrix(lab_m, pred_m_tuned_B)

# # Display confusion matrix with class labels
# disp = ConfusionMatrixDisplay(confusion_matrix=cm,
#                               display_labels=CLASS_NAMES['m'])
# fig, ax = plt.subplots(figsize=(6, 6))
# disp.plot(cmap='Blues', ax=ax, values_format='d')
# plt.title("Confusion Matrix — GatorTron-Base (Model B, tuned)")
# plt.show()

In [ ]:
# # ── Confusion matrices for ensemble predictions ───────────────────────────
# pred_t_final = prob_t_B
# pred_n_final = prob_n_B
# pred_m_final = prob_m_B  # threshold-tuned

# fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# for ax, (p, l, names, task) in zip(axes, [
#     (pred_t_final, lab_t, CLASS_NAMES['t'], 'T14 — Ensemble'),
#     (pred_n_final, lab_n, CLASS_NAMES['n'], 'N03 — Ensemble'),
#     (pred_m_final, lab_m, CLASS_NAMES['m'], 'M01 — Ensemble'),
# ]):
#     cm  = confusion_matrix(l, p)
#     pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
#     ann = np.array([[f'{cm[i,j]}\n({pct[i,j]:.0f}%)'
#                      for j in range(cm.shape[1])]
#                     for i in range(cm.shape[0])])
#     sns.heatmap(cm, annot=ann, fmt='', ax=ax, cmap='Blues',
#                 xticklabels=names, yticklabels=names)
#     ax.set_title(task)
#     ax.set_xlabel('Predicted')
#     ax.set_ylabel('True')
# plt.tight_layout()
# plt.savefig(f'{OUTPUT_DIR}/conf_mat.png', dpi=100)
# plt.show()


## 16. Generate Submission — Ensemble Inference

In [ ]:
# def predict_submission_single(df_sub, task, tok, max_len, model_name, model_tag,
#                                threshold=0.5):
#     """
#     Load best checkpoint for one model/task, run inference, free GPU.
#     Returns raw probabilities (not final preds) so we can ensemble.
#     """
#     ckpt       = f'{OUTPUT_DIR}/best_{task}_{model_tag}.pt'
#     num_labels = len(LABEL_MAPS[task])

#     model = TNMClassifier(model_name, num_labels, CONFIG['dropout']).to(DEVICE)
#     model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

#     ds = TNMDataset(df_sub['text_clean'], None, tok, max_len)
#     ld = DataLoader(ds, batch_size=CONFIG['batch_size'],
#                     collate_fn=dynamic_pad_collate,
#                     num_workers=CONFIG['num_workers'],
#                     pin_memory=True,
#                  worker_init_fn=worker_init_fn)

#     _, probs, _ = run_inference(model, ld, tta=True)

#     model.cpu()
#     del model
#     free_vram(f'after {task.upper()} [{model_tag}] inference')

#     return probs   # shape: (N, num_classes)


# print('Running submission ensemble inference...')

# # ── Model A inference ───────────────────────────────────────────────────
# print('\n--- Model A (BigBird) ---')
# sub_prob_t_A = predict_submission_single(
#     df_sub, 't', tok_t_A, CONFIG['max_len']['t'], MODEL_A, 'A')
# sub_prob_n_A = predict_submission_single(
#     df_sub, 'n', tok_n_A, CONFIG['max_len']['n'], MODEL_A, 'A')
# sub_prob_m_A = predict_submission_single(
#     df_sub, 'm', tok_m_A, CONFIG['max_len']['m'], MODEL_A, 'A')

# # ── Model B inference ───────────────────────────────────────────────────
# print('\n--- Model B (Longformer) ---')
# sub_prob_t_B = predict_submission_single(
#     df_sub, 't', tok_t_B, CONFIG['max_len']['t'], MODEL_B, 'B')
# sub_prob_n_B = predict_submission_single(
#     df_sub, 'n', tok_n_B, CONFIG['max_len']['n'], MODEL_B, 'B')
# sub_prob_m_B = predict_submission_single(
#     df_sub, 'm', tok_m_B, CONFIG['max_len']['m'], MODEL_B, 'B')

# # ── Ensemble ─────────────────────────────────────────────────────────────
# wA = ENSEMBLE_WEIGHTS['A']
# wB = ENSEMBLE_WEIGHTS['B']

# sub_prob_t_ens = wA * sub_prob_t_A + wB * sub_prob_t_B
# sub_prob_n_ens = wA * sub_prob_n_A + wB * sub_prob_n_B
# sub_prob_m_ens = wA * sub_prob_m_A + wB * sub_prob_m_B

# # Final predictions
# t_preds = [INV_LABEL_MAPS['t'][p] for p in sub_prob_t_ens.argmax(-1)]
# n_preds = [INV_LABEL_MAPS['n'][p] for p in sub_prob_n_ens.argmax(-1)]
# # M: apply tuned threshold
# m_raw   = (sub_prob_m_ens[:, 1] > best_thresh).astype(int)
# m_preds = [INV_LABEL_MAPS['m'][p] for p in m_raw]

# submission = pd.DataFrame({
#     'patient_filename': df_sub['patient_filename'],
#     't': t_preds,
#     'n': n_preds,
#     'm': m_preds,
# })
# submission.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)
# print(f'\nsubmission.csv saved — {len(submission)} rows')
# print(f'  T: {submission["t"].value_counts().sort_index().to_dict()}')
# print(f'  N: {submission["n"].value_counts().sort_index().to_dict()}')
# print(f'  M: {submission["m"].value_counts().sort_index().to_dict()}')
# submission.head(10)


## 17. Save Run Metadata